# 05 — Business Dashboard Support
**Source:** `data/silver/`

**Goal:** Produce all charts, tables, and KPIs needed to support a Power BI / business dashboard.
Each section maps to one dashboard panel.

Sections:
1. Load Data
2. KPI Cards
3. Price Distribution Overview
4. Market by City
5. Price per m² Analysis
6. Property Type Breakdown
7. Listing Volume & Trends
8. Export Aggregated Tables for Power BI

## 1 · Load Data

In [ ]:
import glob
import os
import datetime
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Dashboard colour palette (professional) ───────────────────────────────────
PALETTE = {
    "primary"   : "#1A3C5E",
    "secondary" : "#2E86C1",
    "accent"    : "#F39C12",
    "success"   : "#27AE60",
    "danger"    : "#E74C3C",
    "light"     : "#ECF0F1",
}
sns.set_theme(style="white", palette="Blues_d")
plt.rcParams.update({"figure.dpi": 120, "font.family": "DejaVu Sans"})

BASE_DIR   = os.path.dirname(os.getcwd())
SILVER_DIR = os.path.join(BASE_DIR, "data", "silver")
files = sorted(glob.glob(os.path.join(SILVER_DIR, "avito_clean_*.csv")))
assert files, f"No silver files found in {SILVER_DIR}"

df = pd.read_csv(files[-1])
df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce")

# ── Remove artefact rows ──────────────────────────────────────────────────────
BAD_VILLES = {"COURS ET FORMATIONS", "Cours Et Formations"}
df = df[~df["ville"].isin(BAD_VILLES)].reset_index(drop=True)
df = df[df["prix"].notna()].reset_index(drop=True)

# ── Derive property type from titre ──────────────────────────────────────────
def classify_property(titre):
    if pd.isna(titre):
        return "Autre"
    t = titre.lower()
    if any(k in t for k in ["bureau", "plateau", "local", "professionnel", "commercial", "immeuble"]):
        return "Bureau / Commercial"
    if any(k in t for k in ["villa", "riad"]):
        return "Villa / Riad"
    if "studio" in t:
        return "Studio"
    if "appartement" in t or "appart" in t:
        return "Appartement"
    return "Autre"

if "titre" in df.columns:
    df["property_type"] = df["titre"].apply(classify_property)

print(f"Dataset: {len(df)} listings | {df['ville'].nunique()} cities")
print(f"Price range: {df['prix'].min():,.0f} – {df['prix'].max():,.0f} DH/month")

## 2 · KPI Cards

In [ ]:
# ── Compute KPIs ──────────────────────────────────────────────────────────────
kpis = {
    "Total Listings"          : len(df),
    "Cities Covered"          : df["ville"].nunique(),
    "Avg Price (DH/month)"    : round(df["prix"].mean(), 0),
    "Median Price (DH/month)" : round(df["prix"].median(), 0),
    "Avg Surface (m²)"        : round(df["surface_m2"].mean(), 1) if "surface_m2" in df.columns else "N/A",
    "Avg Prix/m²"             : round(df["prix_par_m2"].mean(), 1) if "prix_par_m2" in df.columns else "N/A",
    "% Grande Ville"          : f"{100 * df['is_grande_ville'].mean():.0f}%" if "is_grande_ville" in df.columns else "N/A",
    "Surface Fill Rate"       : f"{100 * df['surface_m2'].notna().mean():.0f}%" if "surface_m2" in df.columns else "N/A",
}

print("📊 DASHBOARD KPIs\n" + "─" * 40)
for k, v in kpis.items():
    print(f"  {k:<30}: {v:>10}")

# ── Visual KPI cards ──────────────────────────────────────────────────────────
kpi_items = list(kpis.items())
n_cols = 4
n_rows = (len(kpi_items) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows))
axes = axes.flatten()

for i, (label, value) in enumerate(kpi_items):
    ax = axes[i]
    ax.set_facecolor(PALETTE["primary"])
    ax.text(0.5, 0.6, str(value), transform=ax.transAxes,
            ha="center", va="center", fontsize=20, fontweight="bold", color="white")
    ax.text(0.5, 0.2, label, transform=ax.transAxes,
            ha="center", va="center", fontsize=9, color=PALETTE["light"])
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

for j in range(len(kpi_items), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Avito.ma Rental Market — Key Metrics",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 3 · Price Distribution Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── 3a: Price histogram ───────────────────────────────────────────────────────
axes[0].hist(df["prix"].dropna(), bins=15,
             color=PALETTE["secondary"], edgecolor="white", linewidth=0.5)
axes[0].set_title("Price Distribution (DH/month)", fontweight="bold")
axes[0].set_xlabel("DH")
axes[0].set_ylabel("Count")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine(ax=axes[0])

# ── 3b: Prix categorie breakdown ──────────────────────────────────────────────
if "categorie_prix" in df.columns:
    cat_order = ["Très Bas", "Bas", "Moyen", "Élevé", "Luxe"]
    cat_counts = df["categorie_prix"].value_counts().reindex(
        [c for c in cat_order if c in df["categorie_prix"].values])
    cat_colors = ["#3498DB", "#2ECC71", "#F39C12", "#E74C3C", "#8E44AD"]
    axes[1].bar(cat_counts.index, cat_counts.values,
                color=cat_colors[:len(cat_counts)], edgecolor="white")
    axes[1].set_title("Listings by Price Category", fontweight="bold")
    axes[1].set_xlabel("Category")
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis="x", rotation=20)
    sns.despine(ax=axes[1])

# ── 3c: Boxplot — price by region ─────────────────────────────────────────────
if "region_label" in df.columns:
    region_order = df.groupby("region_label")["prix"].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df, x="region_label", y="prix",
        order=region_order, ax=axes[2],
        palette="Blues", width=0.5, fliersize=3
    )
    axes[2].set_title("Price by Region", fontweight="bold")
    axes[2].set_xlabel("")
    axes[2].set_ylabel("DH/month")
    axes[2].tick_params(axis="x", rotation=35)
    axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    sns.despine(ax=axes[2])

plt.suptitle("Price Overview", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 4 · Market by City

In [ ]:
# ── City-level aggregation ────────────────────────────────────────────────────
city_stats = (
    df.groupby("ville")
    .agg(
        listings      = ("prix", "count"),
        median_prix   = ("prix", "median"),
        mean_prix     = ("prix", "mean"),
        min_prix      = ("prix", "min"),
        max_prix      = ("prix", "max"),
        mean_surface  = ("surface_m2", "mean"),
        mean_prix_m2  = ("prix_par_m2", "mean"),
    )
    .round(0)
    .sort_values("median_prix", ascending=False)
    .reset_index()
)

print("City Market Summary:\n")
print(city_stats.to_string(index=False))

# ── Chart: median price + listing count per city ──────────────────────────────
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

bars = ax1.bar(
    city_stats["ville"], city_stats["median_prix"],
    color=PALETTE["secondary"], edgecolor="white", alpha=0.85, label="Median Price"
)
ax2.plot(
    city_stats["ville"], city_stats["listings"],
    color=PALETTE["accent"], marker="o", linewidth=2, label="Listings"
)

ax1.set_ylabel("Median Price (DH/month)", color=PALETTE["primary"])
ax2.set_ylabel("Number of Listings", color=PALETTE["accent"])
ax1.tick_params(axis="x", rotation=35)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax1.set_title("Median Price & Listing Volume by City", fontsize=13, fontweight="bold")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

sns.despine(ax=ax1, right=False)
plt.tight_layout()
plt.show()

## 5 · Price per m² Analysis

In [ ]:
df_m2 = df[df["prix_par_m2"].notna() & df["surface_m2"].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── 5a: Prix/m² by city ───────────────────────────────────────────────────────
m2_city = (
    df_m2.groupby("ville")["prix_par_m2"]
    .median().sort_values(ascending=False)
)
m2_city.plot(kind="barh", ax=axes[0], color=PALETTE["secondary"], edgecolor="white")
axes[0].set_title("Median Price/m² by City", fontweight="bold")
axes[0].set_xlabel("DH/m²")
axes[0].set_ylabel("")
sns.despine(ax=axes[0])

# ── 5b: Surface vs Prix scatter ───────────────────────────────────────────────
scatter_df = df_m2.dropna(subset=["surface_m2", "prix"])
hue_col = "property_type" if "property_type" in df_m2.columns else "ville"
sns.scatterplot(
    data=scatter_df, x="surface_m2", y="prix",
    hue=hue_col, s=80, alpha=0.8, ax=axes[1]
)
axes[1].set_title("Surface vs Price", fontweight="bold")
axes[1].set_xlabel("Surface (m²)")
axes[1].set_ylabel("Prix (DH/month)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
axes[1].legend(title=hue_col, bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
sns.despine(ax=axes[1])

plt.suptitle("Price per m² Analysis", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6 · Property Type Breakdown

In [ ]:
if "property_type" not in df.columns:
    print("⚠️  property_type column not found — skipping section.")
else:
    type_stats = (
        df.groupby("property_type")
        .agg(
            count        = ("prix", "count"),
            median_prix  = ("prix", "median"),
            mean_surface = ("surface_m2", "mean"),
        )
        .round(0)
        .sort_values("count", ascending=False)
    )

    print("Property Type Breakdown:\n")
    print(type_stats.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Pie chart — listing share
    type_stats["count"].plot(
        kind="pie", ax=axes[0], autopct="%1.0f%%",
        colors=["#2E86C1","#F39C12","#27AE60","#E74C3C","#8E44AD"],
        startangle=90, wedgeprops=dict(edgecolor="white", linewidth=1.5)
    )
    axes[0].set_ylabel("")
    axes[0].set_title("Share of Listings by Type", fontweight="bold")

    # Bar chart — median price per type
    type_stats["median_prix"].sort_values(ascending=False).plot(
        kind="bar", ax=axes[1],
        color=["#2E86C1","#F39C12","#27AE60","#E74C3C","#8E44AD"],
        edgecolor="white"
    )
    axes[1].set_title("Median Price by Property Type", fontweight="bold")
    axes[1].set_ylabel("DH/month")
    axes[1].set_xlabel("")
    axes[1].tick_params(axis="x", rotation=25)
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    sns.despine(ax=axes[1])

    plt.suptitle("Property Type Analysis", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

## 7 · Listing Volume & Scraping Timeline

In [ ]:
# ── Scrape session overview ───────────────────────────────────────────────────
if df["scraped_at"].notna().sum() > 0:
    scrape_min  = df["scraped_at"].min()
    scrape_max  = df["scraped_at"].max()
    duration    = (scrape_max - scrape_min).total_seconds() / 60

    print(f"Scrape session: {scrape_min.strftime('%Y-%m-%d %H:%M')} → {scrape_max.strftime('%H:%M')}")
    print(f"Duration      : {duration:.1f} minutes")
    print(f"Speed         : {len(df) / duration:.1f} listings/minute")

    # Timeline chart
    df_sorted = df.sort_values("scraped_at")
    df_sorted["listing_num"] = range(1, len(df_sorted) + 1)

    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(df_sorted["scraped_at"], df_sorted["listing_num"],
            color=PALETTE["secondary"], linewidth=2, marker="o", markersize=4)
    ax.fill_between(df_sorted["scraped_at"], df_sorted["listing_num"],
                    alpha=0.15, color=PALETTE["secondary"])
    ax.set_title("Cumulative Listings Scraped Over Time", fontweight="bold")
    ax.set_xlabel("Scrape Time")
    ax.set_ylabel("Cumulative Listings")
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No scraped_at timestamps available.")

## 8 · Export Aggregated Tables for Power BI

In [ ]:
EXPORT_DIR = os.path.join(BASE_DIR, "data", "dashboard")
os.makedirs(EXPORT_DIR, exist_ok=True)

exports = {}

# ── Table 1: Full clean data with property_type ───────────────────────────────
exports["listings_full"] = df.drop(
    columns=[c for c in ["id", "titre", "loaded_at"] if c in df.columns],
    errors="ignore"
)

# ── Table 2: City stats ───────────────────────────────────────────────────────
exports["city_stats"] = city_stats

# ── Table 3: Price category distribution ──────────────────────────────────────
if "categorie_prix" in df.columns:
    cat_dist = (
        df.groupby(["ville", "categorie_prix"])
        .size()
        .reset_index(name="count")
    )
    exports["price_category_by_city"] = cat_dist

# ── Table 4: Property type stats ──────────────────────────────────────────────
if "property_type" in df.columns:
    exports["property_type_stats"] = type_stats.reset_index()

# ── Table 5: KPI summary ──────────────────────────────────────────────────────
exports["kpi_summary"] = pd.DataFrame(
    [{"metric": k, "value": str(v)} for k, v in kpis.items()]
)

# ── Save all exports ──────────────────────────────────────────────────────────
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
for name, table in exports.items():
    path = os.path.join(EXPORT_DIR, f"{name}_{ts}.csv")
    table.to_csv(path, index=False)
    print(f"  ✅ {name:<30}: {table.shape}  → {os.path.basename(path)}")

print(f"\n📁 All tables exported to: {EXPORT_DIR}")
print("   Ready to import into Power BI or Tableau.")